# (A-1b) Analýza štruktúr dát ako súbory (štruktúry a vzťahy, počet, typy, …), záznamy (štruktúry, počet záznamov, počet atribútov, typy, …)

## Imports

In [ ]:
# noinspection PyPackageRequirements
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.stats import normaltest, anderson
from scipy.stats import shapiro
from scipy.stats import spearmanr
import os
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor 

### Load datasets and make copies

In [ ]:
# Load original datasets
df1_original = pd.read_csv("001/observation.csv", sep='\t')
df2_original = pd.read_csv("001/patient.csv", sep='\t')
df3_original = pd.read_csv("001/station.csv", sep='\t')
info_data = pd.read_csv("001/additional_info/sensor_variable_range.csv", sep='\t')

# Define copy folder and copy filenames
copy_folder = "001/copy"
copy1_file = os.path.join(copy_folder, "observation_copy.csv")
copy2_file = os.path.join(copy_folder, "patient_copy.csv")
copy3_file = os.path.join(copy_folder, "station_copy.csv")

# Delete old copies if folder exists
if os.path.exists(copy_folder):
    shutil.rmtree(copy_folder)

# Create new folder
os.makedirs(copy_folder)

# Make copies in memory and save to folder
df1_copy = df1_original.copy()
df1_copy.to_csv(copy1_file, index=False, sep='\t')

df2_copy = df2_original.copy()
df2_copy.to_csv(copy2_file, index=False, sep='\t')

df3_copy = df3_original.copy()
df3_copy.to_csv(copy3_file, index=False, sep='\t')

# Load copies to work with them
df1 = pd.read_csv(copy1_file, sep='\t')
df2 = pd.read_csv(copy2_file, sep='\t')
df3 = pd.read_csv(copy3_file, sep='\t')

In [ ]:
df1.head()

In [ ]:
df2.head()

In [ ]:
df3.head()

In [ ]:
info_data

We have three datasets

1. DF1 – physiological / sensor data (SpO₂, HR, etc.)

2. DF2 – user / patient metadata (username, job, blood group, etc.)

3. DF3 – location / station / administrative data

# 1.1 A - observation

In [ ]:
df1.head()

In [ ]:
# Shape Observation
df1.shape

In [ ]:
df1.info()

As we can see the dataset does not have any missing values, which is great. Lets check it again.

In [ ]:
df1.isna().sum()
df1.isna().sum().sum() 

The info() was correct

In [ ]:
df1.describe()

In [ ]:
print("Unique SPO2: {}".format(df1['SpO₂'].nunique()))
print("Unique HR: {}".format(df1['HR'].nunique()))
print("Unique PI: {}".format(df1['PI'].nunique()))
print("Unique RR: {}".format(df1['RR'].nunique()))
print("Unique ETCO2: {}".format(df1['EtCO₂'].nunique()))
print("Unique FIO2: {}".format(df1['FiO₂'].nunique()))
print("Unique oximetry: {}".format(df1['oximetry'].nunique()))

We know that the dataset has 12000+ rows. Thus we can say this looks suspisious that each culumn results in the same number of unique values. Maybe there are duplicates.. I am not yet sure if the representation of these data is correct if we need floating points. Maybe we need to know heart rate in N number rather than float number

In [ ]:
df1.isnull().sum()

In the df1 there are no nan values

In [ ]:
print("Total rows:", len(df1))
print("Unique rows:", len(df1.drop_duplicates()))

Check if the data is in specific range

In [ ]:
# Define the expected ranges for each variable
value_ranges = {
    'SpO₂': (95, 100),
    'HR': (60, 100),
    'PI': (0.2, 20),
    'RR': (12, 20),
    'EtCO₂': (35, 45),
    'FiO₂': (21, 100),
    'PRV': (20, 200),
    'BP': (90, 120),  # Assuming systolic mean for simplicity
    'Skin Temperature': (33, 38),
    'PVI': (10, 20),
    'Hb level': (12, 18),
    'SV': (60, 100),
    'CO': (4, 8),
    'Signal Quality Index': (0, 100),
    'O₂ extraction ratio': (0.2, 0.3),
    'SNR': (20, 40)
}

# Loop through the variables and check for values outside the expected range
for col, (low, high) in value_ranges.items():
    if col in df1.columns:
        out_of_range = df1[(df1[col] < low) | (df1[col] > high)]
        print(f"{col}: {len(out_of_range)} values outside the range ({low}-{high})")

# We need to check if each sensor data is from any specific station, if not probably we dont want it since it can be missleading
- will be done in A2 after transforming the data

No duplicate records.

## **DF1**
- has 12 072 unique rows with 23 attributes
- each record represents one timestamped observation of physiological measurements
- every attribute is float64 without missing values
- The attributes are physiologically dependent
- Values are in desired range

# 1.1 A - patient

In [ ]:
df2.head()

In [ ]:
df2.shape

In [ ]:
df2.info()

Residence seems off. Also it has 2094 missing values. We can remove this column since it has no value for us.
Otherwise good data types. Some nan values for few attributes.

In [ ]:
df2.isna().sum()

In [ ]:
df2.isna().sum().sum() 

In [ ]:
df2.duplicated().sum()

In [ ]:
# Check overall duplicates (any row identical)
print("Overall duplicate rows:", df2.duplicated().sum())

# Check duplicates by user_id
print("Duplicate user_id values:", df2['user_id'].duplicated().sum())

# Check duplicates by ssn
print("Duplicate SSN values:", df2['ssn'].duplicated().sum())



In [ ]:
df2[df2['user_id'].duplicated(keep=False)].sort_values('user_id')


In [ ]:
# Define valid blood groups
valid_blood_groups = ['A+', 'A-', 'B+', 'B-', 'AB+', 'AB-', 'O+', 'O-']

# Find rows where blood_group is NOT valid
wrong_val = df2[~df2['blood_group'].isin(valid_blood_groups)]

print(f"Number of wrong/invalid blood group values: {len(wrong_val)}")
display(wrong_val[['blood_group']])


# **DF2**
- It has 2094 number of records with 13 attributes
- Residence attribute has no entry, all is nan
- No duplicates
- Some missing values
- We will need to remove one column and fill others with unknown for example
- Current location doesnt follow the same format
- We need to adjust so that each attribute has one format
- Registration has time and data we will stick to date since others does not have date
- Current location is needed to be changed to other format
- Somehow there is duplicates of userid
- Since userid does not meat criteria we will use ssn as identifier because it has no duplicates
- The blood types are correct

# 1.1 A - station

In [ ]:
df3.shape

In [ ]:
df3.head()

In [ ]:
df3.info()

In [ ]:
df3.isna().sum()

In [ ]:
df3.duplicated().sum()

In [ ]:
# Check overall duplicates (any row identical)
print("Overall duplicate rows:", df3.duplicated().sum())

# Check duplicated stations
duplicates_station_coords = df3[df3.duplicated(subset=['station', 'longitude', 'latitude', 'location'], keep=False)]
print(f"Number of true duplicate station records: {duplicates_station_coords.shape[0]}")
display(duplicates_station_coords.head())

duplicates_coords = df3[df3.duplicated(subset=['longitude', 'latitude'], keep=False)]
print(f"Number of repeated coordinates: {duplicates_coords.shape[0]}")
display(duplicates_coords.head())


# DF3
- revision does not have similiar format
- 806 records with 6 attributes
- Only one nan vale in code. Maybe not that important
- No duplicated rows
- There are more entries for specific station maybe we should keep the oldest based on revision
- Each attribute is right data type however revision is needed to be one format

# 1.1A 

# 1.2 A - station

In [ ]:
# Keep only one row per station coordinates (longitude + latitude)
dff3 = df3.drop_duplicates(subset=['station', 'longitude', 'latitude', 'location'], keep='first')

# Drop the revision column
dff3 = dff3.drop(columns=['revision'])


In [ ]:
dff3 = dff3.reset_index(drop=True)

print(dff3.info())

Now the dataframe has less values and keeps only important attributes

In [ ]:
dff3.head()

In [ ]:
dff3.shape

# 1.2 A - patient

In [ ]:
# Drop residence column
dff2 = df2.drop(columns=['residence'])

import re

def extract_lat(x):
    if pd.notnull(x):
        match = re.findall(r"Decimal\('([-\d.]+)'\)", str(x))
        return float(match[0]) if match else None
    return None

def extract_lon(x):
    if pd.notnull(x):
        match = re.findall(r"Decimal\('([-\d.]+)'\)", str(x))
        return float(match[1]) if len(match) > 1 else None
    return None

dff2['current_latitude'] = dff2['current_location'].apply(extract_lat)
dff2['current_longitude'] = dff2['current_location'].apply(extract_lon)

# drop the original column
dff2 = dff2.drop(columns=['current_location'])

# we dont care about address, name, also user_id is duplicated we will use ssn rather, username also not important, job and company can be kept. mail also not important as registration

dff2 = dff2.drop(columns=['username', 'user_id', 'name', 'mail', 'registration'])
# converting current_location to the right format and into two columns
dff2.head()



In [ ]:
dff2.shape

# 1.2 A - observation

In [ ]:
# try to map locations if each observation can be mapped to specific station
merged = df1.merge(dff3, on=['latitude', 'longitude'], how='left', indicator=True)

# Count how many observations have matching locations
print("Mapping Results:")
print(merged['_merge'].value_counts())
print(f"\nTotal observations in df1: {len(df1)}")
print(f"Matched to a station: {len(merged[merged['_merge'] == 'both'])}")
print(f"NOT matched to a station: {len(merged[merged['_merge'] == 'left_only'])}")

# Show percentage
matched_pct = (len(merged[merged['_merge'] == 'both']) / len(df1)) * 100
print(f"\nPercentage matched: {matched_pct:.2f}%")

# Check unique locations in each dataset
print(f"\nUnique locations in df1: {df1[['latitude', 'longitude']].drop_duplicates().shape[0]}")
print(f"Unique locations in dff3: {dff3[['latitude', 'longitude']].drop_duplicates().shape[0]}")

# Count how many unique stations from dff3 were mapped to df1
mapped_stations = merged.loc[merged['_merge'] == 'both', ['latitude', 'longitude']].drop_duplicates()
mapped_station_count = mapped_stations.shape[0]

print(f"\nStations from dff3 mapped to df1: {mapped_station_count} out of {dff3[['latitude', 'longitude']].drop_duplicates().shape[0]}")
print(f"Percentage of stations mapped: {(mapped_station_count / dff3[['latitude', 'longitude']].drop_duplicates().shape[0]) * 100:.2f}%")


We have lost only one station which was the one with no code. And every station is linked correctly with observation data.
Now we can adjust data not to be that precise

Renaming column names so its easier to access

In [ ]:
dff1 = df1.rename(columns={'SpO₂': 'SPO2', 'EtCO₂': 'ETCO2', 'FiO₂': 'FIO2', 'Skin Temperature': 'ST', 'Motion/Activity index' : 'MAI'})

In [ ]:
print("\nColumns in df1:")
print(dff1.columns.tolist())

# 1.2 B - observation

The data does not contain any abnormal values

# 1.2 C - observation

Outlier detection is going to be done with two methods:

1. IQR rule
2. Quantile Trimming

# 1.1 B - observation

In [ ]:
print("Unique SPO2: {}".format(dff1['SPO2'].nunique()))
print("Unique HR: {}".format(dff1['HR'].nunique()))
print("Unique PI: {}".format(dff1['PI'].nunique()))
print("Unique RR: {}".format(dff1['RR'].nunique()))
print("Unique ETCO2: {}".format(dff1['ETCO2'].nunique()))
print("Unique FIO2: {}".format(dff1['FIO2'].nunique()))

The sensors give us precise data.


In [ ]:
print("Rounded unique SPO2:", dff1['SPO2'].round(0).nunique())
print("Rounded unique HR:", dff1['HR'].round(0).nunique())
print("Rounded unique RR:", dff1['RR'].round(0).nunique())
print("Rounded unique EtCO₂:", dff1['ETCO2'].round(0).nunique())

We are going to choose at leat 10 features. Our target is to predict oximetry (which is 0 or 1). Feature types are only continuous. Our data have nonlinear or normal distribution.
Since our target is oximetry and the oximetry is either 1 or 0 we have categorical target.

We need to select 10 attributes which are clinically meaningful, numerical and not categorical. The most meaningful dataset for us is the DF1.

Thats why we are analying these 10 attributes:
1. SpO₂, values 0-100 i think. like we dont need decimal numbers
2. HR	we dont need decimal numbers
3.	PI	we need decimal
4.	RR	not decimal
5.	EtCO₂	no decimal
6.	Motion/Activity index no range
7.	PRV	not decimal
8.	BP	not decimal
9.	Skin Temperature not decimal
10.	CO	not decimal

In [ ]:
df1[["SpO₂","HR","PI","RR","EtCO₂","Motion/Activity index","PRV","BP","Skin Temperature","CO", 'oximetry']].describe()

In [ ]:
numeric_cols = ['SpO₂','HR','PI','RR','EtCO₂','Motion/Activity index','PRV','BP','Skin Temperature','CO']

for col in numeric_cols:
    # Create figure with 2 subplots stacked vertically
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    # Top plot: Histogram with KDE
    sns.histplot(df1[col], bins=40, kde=True, ax=ax1)
    
    # Calculate statistics
    mean_val = df1[col].mean()
    median_val = df1[col].median()
    q1 = df1[col].quantile(0.25)
    q3 = df1[col].quantile(0.75)
    
    # Add vertical lines for mean, median, Q1, Q3 on histogram
    ax1.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
    ax1.axvline(median_val, color='green', linestyle='-', linewidth=2, label=f'Median: {median_val:.2f}')
    ax1.axvline(q1, color='orange', linestyle=':', linewidth=1.5, label=f'Q1: {q1:.2f}')
    ax1.axvline(q3, color='orange', linestyle=':', linewidth=1.5, label=f'Q3: {q3:.2f}')
    ax1.set_title(f'Distribution of {col}')
    ax1.legend()
    
    # Bottom plot: Boxplot (horizontal)
    sns.boxplot(x=df1[col], ax=ax2)
    ax2.set_xlabel(col)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"\n--- {col} ---")
    print(df1[col].describe())  
    print(f"Skewness: {df1[col].skew():.3f}")
    print(f"Kurtosis: {df1[col].kurt():.3f}")
    
    # Normality test
    stat, p = normaltest(df1[col].dropna())
    if p > 0.05:
        print("✅ Probably normal")
    else:
        print("⚠️ Probably not normal")
    
    # Range checks
    if col == 'SpO₂':
        invalid = dff1[(dff1[col] < 95) | (dff1[col] > 100)]
        print(f"Values outside normal range (95–100%): {len(invalid)}")
    elif col == 'HR':
        invalid = dff1[(dff1[col] < 60) | (dff1[col] > 100)]
        print(f"Values outside normal range (60–100 bpm): {len(invalid)}")
    elif col == 'RR':
        invalid = dff1[(dff1[col] < 12) | (dff1[col] > 20)]
        print(f"Values outside normal range (12–20 breaths/min): {len(invalid)}")
    elif col == 'BP':
        invalid = dff1[(dff1[col] < 90) | (dff1[col] > 120)]
        print(f"Values outside normal systolic range (90–120 mmHg): {len(invalid)}")
    elif col == 'Skin Temperature':
        invalid = dff1[(dff1[col] < 33) | (dff1[col] > 38)]
        print(f"Values outside normal range (33–38 °C): {len(invalid)}")
    elif col == 'EtCO₂':
        invalid = dff1[(dff1[col] < 35) | (dff1[col] > 45)]
        print(f"Values outside normal range (35–45 mmHg): {len(invalid)}")
    elif col == 'FiO₂':
        invalid = dff1[(dff1[col] < 21) | (dff1[col] > 100)]
        print(f"Values outside normal range (21–100%): {len(invalid)}")
    elif col == 'PI':
        invalid = dff1[(dff1[col] < 0.2) | (dff1[col] > 20)]
        print(f"Values outside normal range (0.2–20%): {len(invalid)}")
    elif col == 'PRV':
        invalid = dff1[(dff1[col] < 20) | (dff1[col] > 200)]
        print(f"Values outside normal range (20–200 ms): {len(invalid)}")
    elif col == 'CO':
        print(f"Values outside normal range (4–8 L/min): {len(invalid)}")
    print("-" * 60)

1. SpO₂ - bimodial distribution
2. HR - normal distribution
3. PI - left scewed
4. RR - normal distribution
5. EtCO₂ - slightly left scewed
6. Motion/Activity index - bimodal
7. PRV - normal distribution
8. BP - follows normal distribution
9. Skin Temperature - symetrical
10. CO - exponential

# 1.2 C - observation

# IQR function

In [ ]:
# works well for skewed
def iqr_outliers(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Boolean mask of outliers
    outlier_mask = (data < lower_bound) | (data > upper_bound)
    outlier_count = outlier_mask.sum()
    outlier_indices = data[outlier_mask].index.tolist()

    return {

        "outliers": outlier_count,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "indices": outlier_indices
    }

# Standard deviation function


In [ ]:
# normal distribution
def standard_deviation_outliers(data, threshold = 3):
    mean_value = data.mean()
    std_dev = data.std(ddof=1)  # sample std deviation

    lower_bound = mean_value - threshold * std_dev
    upper_bound = mean_value + threshold * std_dev

    outlier_mask = (data < lower_bound) | (data > upper_bound)
    outlier_count = outlier_mask.sum()
    outlier_indices = data[outlier_mask].index.tolist()

    return {
        "outliers": outlier_count,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "indices": outlier_indices
    }


In [ ]:
# Detecting outliers and changing them for bounding values

# SPO2 - distribution is not normal therefore we use IQR function for detecting outliers
iqr_result_spo2 = iqr_outliers(dff1['SPO2'])
print("Number of outliers for SPO2 using IQR rule {}".format(iqr_result_spo2['outliers']))
lower = 0
upper = 0
for index in iqr_result_spo2['indices']:
    if dff1.loc[index, 'SPO2'] < iqr_result_spo2['lower_bound']:
        lower += 1
    else:
        upper += 1
print("Data lower than lower bound({}): {}".format(iqr_result_spo2['lower_bound'], lower))
print("Data bigger than upper bound({}): {}".format(iqr_result_spo2['upper_bound'], upper))
print("Outliers:\n {}".format(dff1.loc[iqr_result_spo2['indices'], 'SPO2']))
if iqr_result_spo2['outliers'] > 0:
    print("Data were caped")
else:
    print("No outlier detected nor deleted")

print("\n\n")

# HR has normal distribution therefore we will use Standard deviation outlier detection
sd_result_hr = standard_deviation_outliers(dff1['HR'], 2)
print("Number of outliers for HR using standard deviation rule {}".format(sd_result_spo2['outliers']))
lower = 0
upper = 0
for index in sd_result_hr['indices']:
    if dff1.loc[index, 'HR'] < sd_result_hr['lower_bound']:
        lower += 1
    else:
        upper += 1
print("Data lower than lower bound({}): {}".format(sd_result_hr['lower_bound'], lower))
print("Data bigger than upper bound({}): {}".format(sd_result_hr['upper_bound'], upper))
print("Outliers:\n {}".format(dff1.loc[sd_result_hr['indices'], 'HR']))
if sd_result_hr['outliers'] > 0:
    print("Data were caped")
else:
    print("No outlier detected nor deleted")

print("\n\n")


# PI - distribution is not normal therefore we use IQR function for detecting outliers
iqr_result_pi = iqr_outliers(dff1['PI'])
print("Number of outliers for PI using IQR rule {}".format(iqr_result_pi['outliers']))
lower = 0
upper = 0
for index in iqr_result_pi['indices']:
    if dff1.loc[index, 'PI'] < iqr_result_pi['lower_bound']:
        lower += 1
    else:
        upper += 1
print("Data lower than lower bound({}): {}".format(iqr_result_pi['lower_bound'], lower))
print("Data bigger than upper bound({}): {}".format(iqr_result_pi['upper_bound'], upper))
print("Outliers:\n {}".format(dff1.loc[iqr_result_pi['indices'], 'PI']))
if iqr_result_pi['outliers'] > 0:
    print("Data were caped")
else:
    print("No outlier detected nor deleted")

print("\n\n")

# RR has normal distribution therefore we will use Standard deviation outlier detection
sd_result_rr = standard_deviation_outliers(dff1['RR'])
print("Number of outliers for RR using standard deviation rule {}".format(sd_result_rr['outliers']))
lower = 0
upper = 0
for index in sd_result_rr['indices']:
    if dff1.loc[index, 'RR'] < sd_result_rr['lower_bound']:
        lower += 1
    else:
        upper += 1
print("Data lower than lower bound({}): {}".format(sd_result_rr['lower_bound'], lower))
print("Data bigger than upper bound({}): {}".format(sd_result_rr['upper_bound'], upper))
print("Outliers:\n {}".format(dff1.loc[sd_result_rr['indices'], 'RR']))
if sd_result_rr['outliers'] > 0:
    print("Data were caped")
else:
    print("No outlier detected nor deleted")

print("\n\n")

# ETCO2 - distribution is slightly skewed therefore we use IQR function for detecting outliers
iqr_result_etco2 = iqr_outliers(dff1['ETCO2'])
print("Number of outliers for ETCO2 using IQR rule {}".format(iqr_result_etco2['outliers']))
lower = 0
upper = 0
for index in iqr_result_etco2['indices']:
    if dff1.loc[index, 'ETCO2'] < iqr_result_etco2['lower_bound']:
        lower += 1
    else:
        upper += 1
print("Data lower than lower bound({}): {}".format(iqr_result_etco2['lower_bound'], lower))
print("Data bigger than upper bound({}): {}".format(iqr_result_etco2['upper_bound'], upper))
print("Outliers:\n {}".format(dff1.loc[iqr_result_etco2['indices'], 'ETCO2']))
if iqr_result_etco2['outliers'] > 0:
    print("Data were caped")
else:
    print("No outlier detected nor deleted")

print("\n\n")

# MAI - distribution is bimodal therefore we use IQR function for detecting outliers
iqr_result_mai = iqr_outliers(dff1['MAI'])
print("Number of outliers for MAI using IQR rule {}".format(iqr_result_mai['outliers']))
lower = 0
upper = 0
for index in iqr_result_mai['indices']:
    if dff1.loc[index, 'MAI'] < iqr_result_mai['lower_bound']:
        lower += 1
    else:
        upper += 1
print("Data lower than lower bound({}): {}".format(iqr_result_mai['lower_bound'], lower))
print("Data bigger than upper bound({}): {}".format(iqr_result_mai['upper_bound'], upper))
print("Outliers:\n {}".format(dff1.loc[iqr_result_mai['indices'], 'MAI']))
if iqr_result_mai['outliers'] > 0:
    print("Data were caped")
else:
    print("No outlier detected nor deleted")

print("\n\n")

# PRV has normal distribution therefore we will use Standard deviation outlier detection
sd_result_prv = standard_deviation_outliers(dff1['PRV'])
print("Number of outliers for PRV using standard deviation rule {}".format(sd_result_prv['outliers']))
lower = 0
upper = 0
for index in sd_result_prv['indices']:
    if dff1.loc[index, 'PRV'] < sd_result_prv['lower_bound']:
        lower += 1
    else:
        upper += 1
print("Data lower than lower bound({}): {}".format(sd_result_prv['lower_bound'], lower))
print("Data bigger than upper bound({}): {}".format(sd_result_prv['upper_bound'], upper))
print("Outliers:\n {}".format(dff1.loc[sd_result_prv['indices'], 'PRV']))
if sd_result_prv['outliers'] > 0:
    print("Data were caped")
else:
    print("No outlier detected nor deleted")

print("\n\n")

# BP has normal distribution therefore we will use Standard deviation outlier detection
sd_result_bp = standard_deviation_outliers(dff1['BP'])
print("Number of outliers for BP using standard deviation rule {}".format(sd_result_bp['outliers']))
lower = 0
upper = 0
for index in sd_result_bp['indices']:
    if dff1.loc[index, 'BP'] < sd_result_bp['lower_bound']:
        lower += 1
    else:
        upper += 1
print("Data lower than lower bound({}): {}".format(sd_result_bp['lower_bound'], lower))
print("Data bigger than upper bound({}): {}".format(sd_result_bp['upper_bound'], upper))
print("Outliers:\n {}".format(dff1.loc[sd_result_bp['indices'], 'BP']))
if sd_result_bp['outliers'] > 0:
    print("Data were cap")
else:
    print("No outlier detected nor deleted")

print("\n\n")

# ST - distribution is not normal therefore we use IQR function for detecting outliers
iqr_result_st = iqr_outliers(dff1['ST'])
print("Number of outliers for ST using IQR rule {}".format(iqr_result_st['outliers']))
lower = 0
upper = 0
for index in iqr_result_st['indices']:
    if dff1.loc[index, 'ST'] < iqr_result_st['lower_bound']:
        lower += 1
    else:
        upper += 1
print("Data lower than lower bound({}): {}".format(iqr_result_st['lower_bound'], lower))
print("Data bigger than upper bound({}): {}".format(iqr_result_st['upper_bound'], upper))
print("Outliers:\n {}".format(dff1.loc[iqr_result_st['indices'], 'ST']))
if iqr_result_st['outliers'] > 0:
    print("Data were caped")
else:
    print("No outlier detected nor deleted")

print("\n\n")

# CO - distribution is not normal therefore we use IQR function for detecting outliers
iqr_result_CO = iqr_outliers(dff1['CO'])
print("Number of outliers for CO using IQR rule {}".format(iqr_result_CO['outliers']))
lower = 0
upper = 0
for index in iqr_result_CO['indices']:
    if dff1.loc[index, 'CO'] < iqr_result_CO['lower_bound']:
        lower += 1
    else:
        upper += 1
print("Data lower than lower bound({}): {}".format(iqr_result_CO['lower_bound'], lower))
print("Data bigger than upper bound({}): {}".format(iqr_result_CO['upper_bound'], upper))
print("Outliers:\n {}".format(dff1.loc[iqr_result_CO['indices'], 'CO']))
if iqr_result_CO['outliers'] > 0:
    print("Data were caped")
else:
    print("No outlier detected nor deleted")

print("\n\n")


def winsorize_outliers(df, column, method='IQR', sd_threshold=3, lower_pct=0.05, upper_pct=0.95, softness=0.99):
    """
    Soft Winsorize outliers in a column to avoid extreme tower effects.

    Parameters:
        df (pd.DataFrame): dataset
        column (str): column name
        method (str): 'IQR' or 'SD' for detection
        sd_threshold (float): threshold for SD method
        lower_pct (float): lower percentile bound
        upper_pct (float): upper percentile bound
        softness (float): how close capped values move toward the bounds (1 = hard cap)

    Returns:
        df (pd.DataFrame): modified dataframe
        summary (dict): summary stats
    """

    data = df[column].copy()

    # 1. Determine hard IQR/SD bounds (for reporting)
    if method == 'IQR':
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        lower_hard = Q1 - 1.5 * IQR
        upper_hard = Q3 + 1.5 * IQR
    elif method == 'SD':
        mean = data.mean()
        std = data.std(ddof=1)
        lower_hard = mean - sd_threshold * std
        upper_hard = mean + sd_threshold * std
    else:
        raise ValueError("Method must be 'IQR' or 'SD'")

    # 2. Get soft Winsorization bounds
    lower_bound = data.quantile(lower_pct)
    upper_bound = data.quantile(upper_pct)

    # 3. Soft cap (not replace by same value, but pull toward bounds)
    capped_data = data.copy()
    capped_data[data < lower_bound] = lower_bound + softness * (data[data < lower_bound] - lower_bound)
    capped_data[data > upper_bound] = upper_bound - softness * (upper_bound - data[data > upper_bound])

    # 4. Compute summary
    lower_count = (data < lower_hard).sum()
    upper_count = (data > upper_hard).sum()
    outliers_count = lower_count + upper_count

    df[column] = capped_data

    summary = {
        'outliers': int(outliers_count),
        'lower_bound': float(lower_bound),
        'upper_bound': float(upper_bound),
        'lower_count': int(lower_count),
        'upper_count': int(upper_count)
    }

    # 5. Print summary
    print(f"Column: {column}")
    print(f"  Total outliers: {outliers_count}")
    print(f"  Lower bound ({lower_bound:.3f}): {lower_count} values adjusted")
    print(f"  Upper bound ({upper_bound:.3f}): {upper_count} values adjusted")
    print(f"  Method: {method} | Softness: {softness}")
    print("  ✅ Soft Winsorization applied.\n")

    return df, summary

# Example usage
features = {
    'SPO2': 'IQR',
    'HR': 'SD',
    'PI': 'IQR',
    'RR': 'SD',
    'ETCO2': 'IQR',
    'MAI': 'IQR',
    'PRV': 'SD',
    'BP': 'SD',
    'ST': 'IQR',
    'CO': 'IQR'
}

for col, method in features.items():
    dff1, _ = winsorize_outliers(dff1, col, method)


In [ ]:
numeric_cols = ['SPO2','HR','PI','RR','ETCO2','MAI','PRV','BP','ST','CO']

for col in numeric_cols:
    # Create figure with 2 subplots stacked vertically
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    # Top plot: Histogram with KDE
    sns.histplot(dff1[col], bins=40, kde=True, ax=ax1)
    
    # Calculate statistics
    mean_val = dff1[col].mean()
    median_val = dff1[col].median()
    q1 = dff1[col].quantile(0.25)
    q3 = dff1[col].quantile(0.75)
    
    # Add vertical lines for mean, median, Q1, Q3 on histogram
    ax1.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
    ax1.axvline(median_val, color='green', linestyle='-', linewidth=2, label=f'Median: {median_val:.2f}')
    ax1.axvline(q1, color='orange', linestyle=':', linewidth=1.5, label=f'Q1: {q1:.2f}')
    ax1.axvline(q3, color='orange', linestyle=':', linewidth=1.5, label=f'Q3: {q3:.2f}')
    ax1.set_title(f'Distribution of {col}')
    ax1.legend()
    
    # Bottom plot: Boxplot (horizontal)
    sns.boxplot(x=dff1[col], ax=ax2)
    ax2.set_xlabel(col)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"\n--- {col} ---")
    print(df1[col].describe())  
    print(f"Skewness: {dff1[col].skew():.3f}")
    print(f"Kurtosis: {dff1[col].kurt():.3f}")
    
    # Normality test
    stat, p = normaltest(dff1[col].dropna())
    if p > 0.05:
        print("✅ Probably normal")
    else:
        print("⚠️ Probably not normal")

    # Range checks
    if col == 'SpO₂':
        invalid = dff1[(dff1[col] < 95) | (dff1[col] > 100)]
        print(f"Values outside normal range (95–100%): {len(invalid)}")
    elif col == 'HR':
        invalid = dff1[(dff1[col] < 60) | (dff1[col] > 100)]
        print(f"Values outside normal range (60–100 bpm): {len(invalid)}")
    elif col == 'RR':
        invalid = dff1[(dff1[col] < 12) | (dff1[col] > 20)]
        print(f"Values outside normal range (12–20 breaths/min): {len(invalid)}")
    elif col == 'BP':
        invalid = dff1[(dff1[col] < 90) | (dff1[col] > 120)]
        print(f"Values outside normal systolic range (90–120 mmHg): {len(invalid)}")
    elif col == 'Skin Temperature':
        invalid = dff1[(dff1[col] < 33) | (dff1[col] > 38)]
        print(f"Values outside normal range (33–38 °C): {len(invalid)}")
    elif col == 'EtCO₂':
        invalid = dff1[(dff1[col] < 35) | (dff1[col] > 45)]
        print(f"Values outside normal range (35–45 mmHg): {len(invalid)}")
    elif col == 'FiO₂':
        invalid = dff1[(dff1[col] < 21) | (dff1[col] > 100)]
        print(f"Values outside normal range (21–100%): {len(invalid)}")
    elif col == 'PI':
        invalid = dff1[(dff1[col] < 0.2) | (dff1[col] > 20)]
        print(f"Values outside normal range (0.2–20%): {len(invalid)}")
    elif col == 'PRV':
        invalid = dff1[(dff1[col] < 20) | (dff1[col] > 200)]
        print(f"Values outside normal range (20–200 ms): {len(invalid)}")
    elif col == 'CO':
        print(f"Values outside normal range (4–8 L/min): {len(invalid)}")
    print("-" * 60)

1. SPO2 - bimodal
2. HR - normal
3. PI - left skewed
4. RR - normal
5. ETCO2 - slightly skewed
6. MAI - right skewed
7. PRV - normal
8. BP - normal
9. ST - symetric
10. CO - exponential

# 1.1 C

### Identify relationships between attributes = Dependencies e.g. correlations 
**Bivariate analysis = Pair analysis**

In [ ]:
dff1.head()

### Pearson correlation heat map for normal distributions and spearman for all

In [ ]:
# List of features
normal_features = ['HR','RR','PRV','BP','ST']
all_features = ['SPO2','HR','PI','RR','ETCO2',
                'MAI','PRV','BP','ST','CO']

# --- 1️⃣ Pearson correlation for normal features ---
corr_pearson = dff1[normal_features].corr(method='pearson')

# Mask upper triangle
mask = np.triu(np.ones_like(corr_pearson, dtype=bool))

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_pearson,
    mask=mask,
    annot=True,
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title('Pearson Correlation (Normal Features)')
plt.show()


# --- 2️⃣ Spearman correlation for all features ---
corr_spearman = dff1[all_features].corr(method='spearman')

# Mask upper triangle
mask = np.triu(np.ones_like(corr_spearman, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_spearman,
    mask=mask,
    annot=True,
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title('Spearman Correlation (All Features)')
plt.show()

HR vs CO: p = +1 -> completely positive linear relationship

ST vs MAI: p = +0.38 -> Moderate positive monotonic correlation

PI vs SPO2: p = +0.36 -> Moderate positive monotonic correlation

ST vs SPO2: p = +0.29 -> Moderate positive monotonic correlation

MAI vs PI: ρ = −0.42 -> Moderate negative correlation

Other variable pairs show very low |ρ| (< 0.1), indicating weak or no monotonic dependency.

Conclusion:

The strongest dependency observed is between HR and CO (positive), and between MAI vs PI -0.42 (negative).

Most physiological variables are relatively independent, supporting their inclusion as separate predictors for modeling oxygen saturation (oximetry).

# 1.1 D

we append a predicted variable to the numeric_col so we can made corelation matrices and compare the values

In [ ]:
numeric_cols.append('oximetry')

### Spearman correlation heatmap with oximetry

In [ ]:
# Compute correlation matrix
corr2 = dff1[numeric_cols].corr(method='spearman')

# Create a mask for the upper triangle
mask2 = np.triu(np.ones_like(corr2, dtype=bool))

# Plot the lower triangular heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr2,
    mask=mask2,          # mask upper part
    annot=True,
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title('Spearman Correlation (monotonic)')
plt.show()

# 1.1 D

## Identify the dependencies between the predicted variable and the other variables (potential predictors)


From Spearman matrices we identified these significant factors:
SpO₂ shows a strong positive correlation with oximetry(0.67).

Perfusion Index (PI) demonstrates a moderate positive correlation(0.4).

Skin temperature exhibits a weaker, yet still notable positive correlation above the defined threshold(0.28).

Blood Preassure exhibits a very week negative correlation(borderline value) however there can be relationship between these values(-0.1).

Since oximetry data contains just two values, we can safely say that the data is not normal(the data are discrete and normal distribution is always continuos). And from normality tests above, we also know that the Sp0₂, PI and Skin Temperature data does not come from normal distribution so we can do the Spearman test.


In [ ]:
def analyze_oximetry_correlations(df):
    pairs = {
        "SpO₂": "SPO2",
        "Perfusion Index": "PI",
        "Skin Temperature": "ST",
        "Blood Pressure": "BP",
        "Motion/Activity Index": "MAI"
    }

    results = []

    print("=== Spearman Correlation Analysis with Oximetry ===\n")

    for label, col in pairs.items():
        rho, p = spearmanr(df['oximetry'], df[col])
        results.append({"Parameter": label, "rho": rho, "p_value": p, "Significant": p < 0.05})

        print(f"Oximetry vs {label}: rho = {rho:.3f}, p = {p:.3e}")
        if p < 0.05:
            print("   → Statistically significant monotonic relationsh")
        else:
            print("   → No statistically significant relationship ")
        print()

    return pd.DataFrame(results)
result_df =analyze_oximetry_correlations(dff1)

In [ ]:
def plot_oximetry_correlations(df):
    plt.figure(figsize=(8, 5))
    bars = plt.barh(df["Parameter"], df["rho"],
                    color=df["Significant"].map({True: "#4CAF50", False: "#B0BEC5"}))

    plt.axvline(0, color='black', linewidth=1)
    plt.xlabel("Spearman correlation (ρ)")
    plt.title("Monotonic correlation with Oximetry")
    plt.grid(axis='x', linestyle='--', alpha=0.5)

    for bar, rho in zip(bars, df["rho"]):
        plt.text(bar.get_width() - 0.06 if rho > 0 else bar.get_width() + 0.06,
                 bar.get_y() + bar.get_height()/2,
                 f"{rho:.3f}",
                 va='center', ha='left' if rho > 0 else 'right', fontsize=9)

    plt.tight_layout()
    plt.show()

plot_oximetry_correlations(result_df)

To evaluate dependencies between the binary target variable oximetry and physiological predictors, we performed the Spearman rank correlation test.
This non-parametric test measures the strength and direction of monotonic (non-linear) relationships and is suitable because oximetry is binary and several predictors are non-normally distributed.

All variables show statistically significant (p < 0.05) monotonic relationships with the target variable oximetry.

SpO₂ is the most dominant factor, which aligns with physiological expectations (direct oxygen saturation measurement).

Perfusion Index and Skin Temperature contribute moderately, suggesting links to peripheral circulation quality.

Blood Pressure and Motion/Activity Index show weak but significant negative trends, possibly due to physiological variability or sensor artifacts.

# 1.1 E

## Document your initial thoughts on solving the project assignment. For example: Are some attributes dependent on each other? Which attributes influence the predicted variable? Is it necessary to combine records from multiple files?

 When looking at the dataset, we need to find out the potential dependencies between attributes and determine which of them could influence the predicted variable, oxygen saturation (SpO₂).

Thus we have calculated both Pearson and Spearman correlation matrixes, to find out tha the Perfusion Index(PI) and Skin Temperature showed the strongest associantons with SpO₂. Although the correlation coefficients were relatively low (ρ ≈ 0.36 for PI and ρ ≈ 0.26 for Skin Temperature), the Spearman test confirmed that these relationships are statistically significant.

The dataset used for this stage of the project contained all required variables within a single file, so combining multiple data sources was not necessary at this point. However, there is potential to cluster patients location to the closest station id.

# 1.3

## Formulate two hypotheses about the data in the context of the given predictive task. Verify the formulated hypotheses using appropriately chosen statistical tests.


Since we know the parameters that are in relationship with predicted value we can form a hypotesis that will help us to further more predict the effects that will be formed between values. Our hypotesis:

## $H_0$ = The variance of PI is the same in the oximetry = 0 group as in the oximetry = 1 group.

## $H_1$ = The variance of PI is not the same in the oximetry = 0  as in the oximetry = 1 group

In [ ]:
hpc1 = df1[['PI','oximetry']]
hpc1.head()

In [ ]:
hpc1.describe()

In [ ]:
hpc1['oximetry'].unique()
counts = hpc1['oximetry'].value_counts()
counts

In [ ]:
counts.plot(kind='bar')
plt.xlabel('')
plt.ylabel('Count')
plt.title('Count of oximetry Values')
plt.show()


In [ ]:
sns.violinplot(x="oximetry", y="PI", data=hpc1)

From the violin above, we can see that the distribution of PI could be a difference between oximetry = 1 group and oximetry = 0 with respect to PI suqquesting a potential difference in variance. However the visual observation should be confirmed with statistical tests.

In [ ]:
hpc1_ox1 = hpc1.loc[hpc1['oximetry'] == 1, ['oximetry', 'PI']]
sns.histplot(data=hpc1_ox1, x='PI')

In [ ]:
hpc1_ox0 = hpc1.loc[hpc1['oximetry'] == 0, ['oximetry', 'PI']]
sns.histplot(data=hpc1_ox1, x='PI')

The PI probably doesn't come from normal distribution so the data split to 2 groups(oximetry=1 and oximetry = 0 groups) doesn't seem to come from normal distribution either, however to verify that we need to do some normality tests

In [ ]:
from scipy.stats import levene

# Split PI values by oximetry group
pi_0 = dff1.loc[dff1['oximetry'] == 0, 'PI']
pi_1 = dff1.loc[dff1['oximetry'] == 1, 'PI']

# Levene's test for equal variances
stat, p = levene(pi_0, pi_1)
print(f"Levene test statistic = {stat:.4f}, p-value = {p:.4e}")

if p < 0.05:
    print("Reject H0 → Variances are significantly different.")
else:
    print("Fail to reject H0 → Variances are not significantly different.")


In [ ]:

hpc1_ox1.shape

In [ ]:
hpc1_ox0.shape
hpc1_ox0.head()

Since we have more